# Hallucination Detection Benchmark — Google Colab Runner

Runs the `benchmark/` pipeline (SelfCheckGPT vs RAG Verification vs LLM-as-a-Judge) on a Colab GPU, since it needs more RAM/VRAM than a typical local machine.

**Before running:** `Runtime -> Change runtime type -> T4 GPU` (or better).

Steps: (1) get the code onto Colab, (2) install dependencies, (3) run the benchmark, (4) view/download results.

In [ ]:
!nvidia-smi

## Step 1 — Get the code onto Colab

**Option A (recommended): upload a zip.** On your local machine, zip the `benchmark/` folder
(the one containing `main.py`, `config.py`, etc.) into `benchmark.zip`, then run the cell below and
pick that file.

If you already used Option A, skip the "Option B" cell further down.

In [ ]:
from google.colab import files

uploaded = files.upload()  # select your benchmark.zip in the dialog

In [ ]:
import os
import zipfile

zip_name = list(uploaded.keys())[0]
print(f"Uploaded file: {zip_name} ({os.path.getsize(zip_name)} bytes)")

with zipfile.ZipFile(zip_name, "r") as z:
    names = z.namelist()
    print(f"Zip contains {len(names)} entries, e.g.: {names[:10]}")
    assert any(n.endswith("main.py") for n in names), (
        "This zip has no main.py in it — re-zip the 'benchmark' folder itself "
        "(the one with main.py/config.py inside it), not its parent or an unrelated folder."
    )
    # Windows' Compress-Archive writes backslashes as path separators, which zipfile
    # on Linux treats as literal filename characters instead of directories -
    # normalize them before extracting.
    for member in z.infolist():
        normalized = member.filename.replace("\\", "/")
        target = os.path.join("/content", normalized)
        if normalized.endswith("/"):
            os.makedirs(target, exist_ok=True)
            continue
        os.makedirs(os.path.dirname(target), exist_ok=True)
        with z.open(member) as src, open(target, "wb") as dst:
            dst.write(src.read())

print(os.listdir("/content"))

**Option B: clone from a git repo instead of uploading a zip.** Only run this if you pushed the
`benchmark/` folder to your own GitHub repo, and skip Option A above. Edit `REPO_URL` first.

In [ ]:
REPO_URL = "https://github.com/<your-username>/<your-repo>.git"  # edit me
!git clone "$REPO_URL" /content/repo

In [ ]:
# Auto-locate main.py under /content instead of assuming a fixed path, since the
# zip/clone can extract with different top-level folder names (or no subfolder at all).
import os

matches = [root for root, _, fnames in os.walk("/content") if "main.py" in fnames and "config.py" in fnames]
assert matches, "Couldn't find main.py under /content — check that the upload/clone step above succeeded."
BENCHMARK_DIR = matches[0]
print(f"Found benchmark code at: {BENCHMARK_DIR}")
%cd $BENCHMARK_DIR

## Step 2 — Install dependencies

This installs everything needed for `--mock`, `--lite`, and `--backend transformers` runs.
`vllm` is intentionally left out here (heavy install, only needed for `--backend vllm`) — see the
optional cell below if you want it.

`torch` is deliberately **not** reinstalled: Colab ships with a GPU-matched build preinstalled, and
`pip install torch` can silently replace it with a mismatched/CPU build and break GPU access.

In [ ]:
!pip install -q transformers sentence-transformers faiss-cpu datasets pandas scipy scikit-learn matplotlib seaborn

In [ ]:
import torch

print(f"torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU visible to torch - check Runtime > Change runtime type > GPU, "
          "then Runtime > Disconnect and delete runtime, then reconnect.")

Optional — only needed for `--backend vllm` with the full 8B paper-scale models. Slow install,
and the 8B models likely won't fit on a free T4 GPU (16GB VRAM). Prefer `--paper` (or `--lite`)
below unless you have a bigger GPU (e.g. Colab Pro A100).


In [ ]:
# !pip install -q vllm

## Step 3 — Run the benchmark

Pick one:
- `--mock`: fastest, no downloads, just sanity-checks the pipeline.
- `--lite` (smallest, works almost anywhere): tiny 0.5B models, for machines/GPUs with very little RAM/VRAM.
- `--paper` (recommended on a free T4): 1.5B generator + judge (sharing weights, no extra VRAM) and a
  small BGE embedder — sized to fit a free single-T4 Colab GPU while staying closer to paper scale.
- `--paper --strong-judge`: as above, plus a second judging pass with a 3B model, loaded only after
  the 1.5B generator/judge are freed (still fits a T4 since the models are never resident together).
- `--backend transformers` (no `--lite`/`--paper`): full paper-scale 8B models — needs a large-VRAM GPU
  (e.g. Colab Pro A100), not a free T4.


In [ ]:
!python main.py --mock --total-prompts 40

In [ ]:
!python main.py --lite --total-prompts 100

In [ ]:
!python main.py --paper --total-prompts 100


In [ ]:
!python main.py --paper --strong-judge --total-prompts 100


In [ ]:
# Full paper-scale run - only if your Colab GPU has enough VRAM for an 8B model.
# !python main.py --backend transformers --total-prompts 500

## Step 4 — View and download results

In [ ]:
import os

import pandas as pd
from IPython.display import display

display(pd.read_csv("results/accuracy_metrics.csv"))

# Only present if you ran with --strong-judge
if os.path.exists("results/strong_judge_accuracy_metrics.csv"):
    display(pd.read_csv("results/strong_judge_accuracy_metrics.csv"))


In [ ]:
from IPython.display import Image, display

display(Image("results/plots/pareto_frontier.png"))
display(Image("results/plots/ttft_distribution.png"))

In [ ]:
import shutil

from google.colab import files

shutil.make_archive("/content/results", "zip", "results")
files.download("/content/results.zip")